# Seed Coding Books into MongoDB with Ollama Embeddings

This notebook seeds one or more books into MongoDB for RAG.

## What it does
- Reads `.txt`, `.md`, and optionally `.pdf` files from a folder
- Splits each book into overlapping word-count chunks
- Generates embeddings from a local Ollama model using `POST /api/embed`
- Stores each chunk plus metadata and the embedding in MongoDB
- Uses upserts so you can rerun safely without creating duplicates

## Suggested collection shape
Each chunk document includes:
- `book_id` — slugified title used as stable key
- `title` — human-readable book title (file stem)
- `author` — configurable metadata
- `chunk_index` — sequential position within the book
- `text` — raw chunk text
- `embedding` — dense float vector from Ollama
- `tags` — free-form labels
- `source_type` — `"book"` by default

## Assumptions
- MongoDB is running and reachable at `MONGO_URI`
- Ollama is running locally with an embedding model already pulled
- Books are plain-text, markdown, or PDF files in `BOOKS_DIR`

> **Tip:** If your books are PDFs, install `pypdf` (see the cell below) for automatic extraction.

## Notes

MongoDB stores vector embeddings alongside document data. The `embedding` field holds a flat
array of floats per chunk. Ollama's `/api/embed` endpoint accepts a single string or an array
of strings and returns an `embeddings` array. PyMongo's `insert_one()` and `insert_many()` are
the standard insert operations for MongoDB documents.

In [2]:
# If needed, uncomment and run this cell once to install dependencies.
# %pip install pymongo requests pypdf

In [30]:
from pathlib import Path
from typing import Iterable, List, Dict, Optional
import hashlib
import re
import requests
from pymongo import MongoClient, ReplaceOne
from pymongo.errors import PyMongoError

try:
    from pypdf import PdfReader
    PDF_ENABLED = True
except Exception:
    PDF_ENABLED = False

## Configuration

Update these values to match your local environment.

In [31]:
# --- MongoDB ---
MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "engineering_copilot"
COLLECTION_NAME = "book_chunks"

# --- Ollama embeddings ---
OLLAMA_EMBED_URL = "http://localhost:11434/api/embed"
EMBED_MODEL = "embeddinggemma"
EMBED_BATCH_SIZE = 16

# --- Input books folder ---
BOOKS_DIR = Path("../books")

# --- Chunking ---
# Keep chunks under ~325 tokens so they fit inside embeddinggemma's 512-token context window.
# At ~1.3 tokens/word: 250 words ≈ 325 tokens (safe margin for punctuation / sub-word splits).
CHUNK_WORDS = 250
CHUNK_OVERLAP_WORDS = 40

# --- Metadata defaults ---
DEFAULT_AUTHOR = "Unknown"
DEFAULT_TAGS = ["book", "coding"]

# --- Source type tag ---
SOURCE_TYPE = "book"

## Helpers: File Loading and Chunking

In [32]:
def slugify(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "-", text)
    return text.strip("-")


def stable_hash(text: str, length: int = 12) -> str:
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:length]


def read_text_file(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")


def read_pdf_file(path: Path) -> str:
    if not PDF_ENABLED:
        raise RuntimeError("pypdf is not installed. Run: %pip install pypdf")
    reader = PdfReader(str(path))
    pages = []
    for page in reader.pages:
        pages.append(page.extract_text() or "")
    return "\n\n".join(pages)


def load_book_text(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix in {".txt", ".md"}:
        return read_text_file(path)
    if suffix == ".pdf":
        return read_pdf_file(path)
    raise ValueError(f"Unsupported file type: {path.suffix}")

In [33]:
def split_words(text: str) -> List[str]:
    return re.findall(r"\S+", text)


def chunk_text(text: str, chunk_words: int = 900, overlap_words: int = 150) -> List[str]:
    words = split_words(text)
    if not words:
        return []

    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_words, len(words))
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)

        if end == len(words):
            break

        start = max(end - overlap_words, 0)

    return chunks

## Helpers: Embeddings with Ollama

Ollama's `/api/embed` endpoint accepts either a single string or an array of strings and
returns an `embeddings` array. This notebook batches multiple chunks per request for better
throughput.

In [34]:
def batched(items: List[str], size: int) -> Iterable[List[str]]:
    for i in range(0, len(items), size):
        yield items[i:i + size]


def embed_batch(texts: List[str], model: str = EMBED_MODEL) -> List[List[float]]:
    resp = requests.post(
        OLLAMA_EMBED_URL,
        json={
            "model": model,
            "input": texts,
        },
        timeout=300,
    )
    resp.raise_for_status()
    data = resp.json()
    return data["embeddings"]


def embed_texts(texts: List[str], batch_size: int = EMBED_BATCH_SIZE) -> List[List[float]]:
    all_embeddings: List[List[float]] = []
    for batch in batched(texts, batch_size):
        all_embeddings.extend(embed_batch(batch))
    return all_embeddings

## MongoDB Setup

Creates the MongoDB client, selects the target collection, and ensures a unique compound
index on `(book_id, chunk_index)` so upserts are idempotent.

In [35]:
client = MongoClient(MONGO_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

# Compound index: one chunk per (book, position) — unique ensures safe upserts
collection.create_index(
    [("book_id", 1), ("chunk_index", 1)],
    unique=True,
    name="book_chunk_idx",
)

print(f"Connected to {MONGO_URI}  |  db={DB_NAME}  |  collection={COLLECTION_NAME}")

Connected to mongodb://localhost:27017  |  db=engineering_copilot  |  collection=book_chunks


## Build Chunk Documents

`build_chunk_docs` takes a single file, chunks it, embeds every chunk, and returns a list of
MongoDB documents ready for upsert. Each document carries a stable `_id` derived from the
book identifier and chunk index so reruns are safe.

In [36]:
def build_chunk_docs(
    file_path: Path,
    author: str = DEFAULT_AUTHOR,
    tags: Optional[List[str]] = None,
    chunk_words: int = CHUNK_WORDS,
    overlap_words: int = CHUNK_OVERLAP_WORDS,
) -> List[Dict]:
    title = file_path.stem
    book_id = slugify(title)
    tags = tags or DEFAULT_TAGS

    text = load_book_text(file_path)
    chunks = chunk_text(text, chunk_words=chunk_words, overlap_words=overlap_words)
    embeddings = embed_texts(chunks)

    docs = []
    for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
        doc = {
            "book_id": book_id,
            "title": title,
            "author": author,
            "chunk_index": i,
            "text": chunk,
            "embedding": embedding,
            "tags": tags,
            "source_type": SOURCE_TYPE,
        }
        docs.append(doc)

    print(f"  {title}: {len(chunks)} chunks")
    return docs

## Upsert Chunk Documents into MongoDB

Uses PyMongo's `bulk_write` with `ReplaceOne(..., upsert=True)` on the compound key
`(book_id, chunk_index)`. This means reruns update existing documents rather than
inserting duplicates.

In [37]:
def upsert_chunk_docs(docs: List[Dict]) -> None:
    if not docs:
        return

    operations = [
        ReplaceOne(
            filter={"book_id": d["book_id"], "chunk_index": d["chunk_index"]},
            replacement=d,
            upsert=True,
        )
        for d in docs
    ]

    try:
        result = collection.bulk_write(operations, ordered=False)
        print(
            f"  Upserted {result.upserted_count}, "
            f"modified {result.modified_count} documents."
        )
    except PyMongoError as exc:
        print(f"  MongoDB error: {exc}")

## Seed All Books from a Folder

Iterates every supported file in `BOOKS_DIR`, builds chunk documents, and upserts them.
Supported extensions: `.txt`, `.md`, and (if `pypdf` is installed) `.pdf`.

In [38]:
SUPPORTED_SUFFIXES = {".txt", ".md"}
if PDF_ENABLED:
    SUPPORTED_SUFFIXES.add(".pdf")


def seed_books_from_folder(folder: Path) -> None:
    if not folder.exists():
        print(f"Folder not found: {folder.resolve()}")
        return

    files = sorted(
        f for f in folder.iterdir()
        if f.is_file() and f.suffix.lower() in SUPPORTED_SUFFIXES
    )

    if not files:
        print(f"No supported files in {folder.resolve()}")
        return

    print(f"Seeding {len(files)} book(s) from {folder.resolve()}\n")

    for file_path in files:
        print(f"Processing: {file_path.name}")
        try:
            docs = build_chunk_docs(file_path)
            upsert_chunk_docs(docs)
        except Exception as exc:
            print(f"  ERROR processing {file_path.name}: {exc}")

    print("\nDone.")

## Run the Seeding Job

Run this cell to process all books in `BOOKS_DIR`.  
Place your `.txt`, `.md`, or `.pdf` files in that folder first.

In [39]:
seed_books_from_folder(BOOKS_DIR)

Seeding 10 book(s) from C:\Users\mario\OneDrive\Desktop\ai_ml_expert\books

Processing: Black Hat Python ( PDFDrive ).pdf
  Black Hat Python ( PDFDrive ): 234 chunks
  Upserted 234, modified 0 documents.
Processing: Deep-Learning-with-PyTorch.pdf
  Deep-Learning-with-PyTorch: 858 chunks
  Upserted 858, modified 0 documents.
Processing: deep-learning-with-tensorflow-20-and-keras.pdf
  deep-learning-with-tensorflow-20-and-keras: 684 chunks
  Upserted 684, modified 0 documents.
Processing: Expert_Python_Programming.pdf
  Expert_Python_Programming: 399 chunks
  Upserted 399, modified 0 documents.
Processing: Grokking Algorithms, 2nd Edition (MEAP).pdf
  Grokking Algorithms, 2nd Edition (MEAP): 116 chunks
  Upserted 116, modified 0 documents.
Processing: java_developer_guide.pdf
  java_developer_guide: 30 chunks
  Upserted 30, modified 0 documents.
Processing: Linear_Algebra.pdf
  Linear_Algebra: 897 chunks
  Upserted 897, modified 0 documents.
Processing: Pandas-Cookbook-eBook.pdf
  Pandas

## Inspect What Was Inserted

Run a quick sanity check — prints the first three chunks and the total count
stored in the collection.

In [41]:
total = collection.count_documents({})
print(f"Total documents in '{COLLECTION_NAME}': {total}\n")

for doc in collection.find({}, {"_id": 0, "text": 1, "book_id": 1, "chunk_index": 1}).limit(3):
    print(f"book_id={doc['book_id']}  chunk_index={doc['chunk_index']}")
    print(f"  text preview: {doc['text'][:120]!r}")
    print()

Total documents in 'book_chunks': 4735

book_id=black-hat-python-pdfdrive  chunk_index=0
  text preview: 'Black Hat Python: Python Programming for Hackers and Pentesters Justin Seitz Published by No Starch Press To Pat Althoug'

book_id=black-hat-python-pdfdrive  chunk_index=1
  text preview: 'employed as a security consultant, doing everything from policy review to penetration tests, and he feels lucky to have '

book_id=black-hat-python-pdfdrive  chunk_index=2
  text preview: 'write network packets, how to sniff the network, as well as anything you might need for web application auditing and att'



## Optional: Query-Time Embedding

Use `embed_query` to embed a search string and retrieve the most similar chunks
using a cosine similarity scan. This is useful for ad-hoc testing before wiring
full vector search.

In [ ]:
def embed_query(text: str) -> List[float]:
    return embed_batch([text])[0]


# Example usage
query_vector = embed_query("how does garbage collection work in Java?")
print(f"Query embedding dimension: {len(query_vector)}")
print(f"First 5 values: {query_vector[:5]}")

## Optional: MongoDB Atlas Vector Search

If you migrate to **MongoDB Atlas**, you can create a Vector Search index and run
`$vectorSearch` aggregation pipelines for ANN retrieval. The cell below shows the pattern
— uncomment and adapt once you have an Atlas cluster with a vector search index configured.

In [ ]:
# Atlas Vector Search example — requires an Atlas cluster with a Vector Search index
# named "embedding_index" on the "embedding" field.
#
# query_vec = embed_query("What is the observer pattern?")
#
# pipeline = [
#     {
#         "$vectorSearch": {
#             "index": "embedding_index",
#             "path": "embedding",
#             "queryVector": query_vec,
#             "numCandidates": 200,
#             "limit": 5,
#         }
#     },
#     {
#         "$project": {
#             "title": 1,
#             "chunk_index": 1,
#             "text": {"$substr": ["$text", 0, 200]},
#             "score": {"$meta": "vectorSearchScore"},
#             "embedding": 0,
#         }
#     },
# ]
#
# results = list(collection.aggregate(pipeline))
# for r in results:
#     print(f"[{r['score']:.4f}] {r['title']} chunk {r['chunk_index']}")
#     print(f"  {r['text']!r}")
#     print()
print("Atlas Vector Search cell — uncomment when ready.")